# Bronze Ingestion

## Imports

In [1]:
from pyspark.sql.functions import current_timestamp, input_file_name, lit

StatementMeta(, ca57a215-98d6-4587-a2eb-2069e8996a1c, 3, Finished, Available, Finished, False)

## Read Raw CFPB Complaints CSV

In [2]:
df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .load("Files/raw/complaints.csv")
)

StatementMeta(, ca57a215-98d6-4587-a2eb-2069e8996a1c, 4, Finished, Available, Finished, False)

In [3]:
display(df.limit(10))

StatementMeta(, ca57a215-98d6-4587-a2eb-2069e8996a1c, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c6ee9735-1e68-49c0-a75e-1e1a0831a57c)

In [4]:
df.printSchema()

StatementMeta(, ca57a215-98d6-4587-a2eb-2069e8996a1c, 6, Finished, Available, Finished, False)

root
 |-- Date received: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Sub-product: string (nullable = true)
 |-- Issue: string (nullable = true)
 |-- Sub-issue: string (nullable = true)
 |-- Consumer complaint narrative: string (nullable = true)
 |-- Company public response: string (nullable = true)
 |-- Company: string (nullable = true)
 |-- State: string (nullable = true)
 |-- ZIP code: string (nullable = true)
 |-- Tags: string (nullable = true)
 |-- Submitted via: string (nullable = true)
 |-- Date sent to company: string (nullable = true)
 |-- Company response to consumer: string (nullable = true)
 |-- Timely response?: string (nullable = true)
 |-- Complaint ID: string (nullable = true)



## Rename Columns

In [5]:
df_renamed = (
    df
    .withColumnRenamed("Date received", "date_received")
    .withColumnRenamed("Product", "product")
    .withColumnRenamed("Sub-product", "sub_product")
    .withColumnRenamed("Issue", "issue")
    .withColumnRenamed("Sub-issue", "sub_issue")
    .withColumnRenamed("Consumer complaint narrative", "consumer_complaint_narrative")
    .withColumnRenamed("Company public response", "company_public_response")
    .withColumnRenamed("Company", "company")
    .withColumnRenamed("State", "state")
    .withColumnRenamed("ZIP code", "zip_code")
    .withColumnRenamed("Tags", "tags")
    .withColumnRenamed("Consumer consent provided?", "consumer_consent_provided")
    .withColumnRenamed("Submitted via", "submitted_via")
    .withColumnRenamed("Date sent to company", "date_sent_to_company")
    .withColumnRenamed("Company response to consumer", "company_response_to_consumer")
    .withColumnRenamed("Timely response?", "timely_response")
    .withColumnRenamed("Consumer disputed?", "consumer_disputed")
    .withColumnRenamed("Complaint ID", "complaint_id")
)

StatementMeta(, ca57a215-98d6-4587-a2eb-2069e8996a1c, 7, Finished, Available, Finished, False)

In [6]:
display(df_renamed.limit(10))

StatementMeta(, ca57a215-98d6-4587-a2eb-2069e8996a1c, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 02a0632a-d578-42dc-8e6f-f2deacce840b)

In [7]:
df_renamed.printSchema()

StatementMeta(, ca57a215-98d6-4587-a2eb-2069e8996a1c, 9, Finished, Available, Finished, False)

root
 |-- date_received: string (nullable = true)
 |-- product: string (nullable = true)
 |-- sub_product: string (nullable = true)
 |-- issue: string (nullable = true)
 |-- sub_issue: string (nullable = true)
 |-- consumer_complaint_narrative: string (nullable = true)
 |-- company_public_response: string (nullable = true)
 |-- company: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- submitted_via: string (nullable = true)
 |-- date_sent_to_company: string (nullable = true)
 |-- company_response_to_consumer: string (nullable = true)
 |-- timely_response: string (nullable = true)
 |-- complaint_id: string (nullable = true)



## Add Bronze Metadata

In [8]:
df_bronze = (
    df_renamed
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", input_file_name())
    .withColumn("batch_id", lit("batch_001"))
)

StatementMeta(, ca57a215-98d6-4587-a2eb-2069e8996a1c, 10, Finished, Available, Finished, False)

In [9]:
display(df_bronze.limit(10))

StatementMeta(, ca57a215-98d6-4587-a2eb-2069e8996a1c, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fcfc182e-9108-445d-8310-8010c2cf6817)

## Create Bronze Schema

In [10]:
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

StatementMeta(, ca57a215-98d6-4587-a2eb-2069e8996a1c, 12, Finished, Available, Finished, False)

DataFrame[]

## Write Bronze Delta Table

In [11]:
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze.complaints")
)

StatementMeta(, ca57a215-98d6-4587-a2eb-2069e8996a1c, 13, Finished, Available, Finished, False)

## Verify Bronze Table

In [12]:
display(spark.sql("SELECT * FROM bronze.complaints LIMIT 10"))

StatementMeta(, ca57a215-98d6-4587-a2eb-2069e8996a1c, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 43bc168b-2d94-45ce-8a45-0b6bc04beb07)

In [13]:
spark.sql("DESCRIBE TABLE bronze.complaints").show(truncate=False)

StatementMeta(, ca57a215-98d6-4587-a2eb-2069e8996a1c, 15, Finished, Available, Finished, False)

+----------------------------+---------+-------+
|col_name                    |data_type|comment|
+----------------------------+---------+-------+
|date_received               |string   |NULL   |
|product                     |string   |NULL   |
|sub_product                 |string   |NULL   |
|issue                       |string   |NULL   |
|sub_issue                   |string   |NULL   |
|consumer_complaint_narrative|string   |NULL   |
|company_public_response     |string   |NULL   |
|company                     |string   |NULL   |
|state                       |string   |NULL   |
|zip_code                    |string   |NULL   |
|tags                        |string   |NULL   |
|submitted_via               |string   |NULL   |
|date_sent_to_company        |string   |NULL   |
|company_response_to_consumer|string   |NULL   |
|timely_response             |string   |NULL   |
|complaint_id                |string   |NULL   |
|ingestion_timestamp         |timestamp|NULL   |
|source_file        

In [14]:
spark.sql("SELECT COUNT(*) AS total_rows FROM bronze.complaints").show()

StatementMeta(, ca57a215-98d6-4587-a2eb-2069e8996a1c, 16, Finished, Available, Finished, False)

+----------+
|total_rows|
+----------+
|  16294931|
+----------+

